# Proyecto EVA · POC v2 — Detección de patrones de cristalización salival ("helechos") con MobileNet

**Objetivo de este notebook:** comprobar si existe *señal aprendible* en fotos espontáneas de saliva tomadas con un microscopio de bolsillo (50x), es decir, si una red neuronal puede distinguir "hay patrón de helecho" de "no hay patrón". **Es una prueba de concepto exploratoria**, no un test clínico.

### Cómo está organizado (se ejecuta de arriba hacia abajo)

| Paso | Qué pasa | ¿Necesita tu intervención? |
|---|---|---|
| 0 | Instalar y configurar | Revisar rutas |
| 1 | Crear automáticamente la planilla (CSV) con todas las fotos | No |
| 2 | **Etiquetar las fotos** (helecho / no helecho) | **Sí, es el paso manual** |
| 3 | Revisar los datos (EDA) y detectar fotos duplicadas | No |
| 4 | Separar datos en *entrenamiento* y *test congelado* | No |
| 5 | Entrenar la red (con validación cruzada) | No (toma unos minutos) |
| 6 | Comparar con un método clásico más simple (línea base) | No |
| 7 | Evaluar una sola vez sobre el test congelado | No |
| 8 | Ver *qué mira* la red (Grad-CAM) | No |
| 9 | Probar con una foto nueva | Poner la ruta de una foto |

### Mini glosario

- **CNN / MobileNet:** tipo de red neuronal para imágenes. MobileNet es una versión liviana, pensada para correr en teléfonos.
- **Transfer learning:** partir de una red que ya aprendió a "ver" (con millones de fotos generales) y solo reajustarla a nuestro problema. Es clave cuando hay pocas imágenes.
- **Época:** una pasada completa por todas las imágenes de entrenamiento.
- **Validación cruzada (CV):** repetir el entrenamiento varias veces, rotando qué parte de los datos se usa para "examinar" al modelo. Da una estimación más estable cuando hay pocos datos.
- **Test congelado:** un grupo de fotos que se aparta al inicio y **no se toca** hasta el final. Es la única medición realmente honesta.
- **Fuga de datos:** cuando fotos casi idénticas quedan en entrenamiento *y* en evaluación; el modelo "copia" en vez de aprender y las métricas salen falsamente buenas.
- **Sensibilidad:** de las fotos que *sí* tienen helecho, qué porcentaje detecta el modelo. **Especificidad:** de las que *no* tienen, qué porcentaje descarta correctamente.
- **AUC:** número entre 0.5 (azar) y 1.0 (perfecto) que resume qué tan bien separa el modelo ambas clases.
- **Grad-CAM:** mapa de calor que muestra en qué zona de la imagen se fijó el modelo para decidir.

> ⚠️ **Marco de interpretación:** los resultados miden "patrón compatible con cristalización en fotos aportadas espontáneamente". **No** miden fertilidad ni ovulación.

## Paso 0 · Instalación y configuración — *dejar el entorno listo y definir las rutas*

> **¿Qué hace?** Instala las librerías extra que se necesitan, importa todo y define los parámetros principales en un solo lugar.
> **¿Qué tienes que hacer?** Solo revisar `IMAGES_DIR` (carpeta donde están las fotos) y `CSV_PATH` (planilla de etiquetas). El resto puede quedar como está.

In [ ]:
# Ejecutar solo la primera vez (en Colab/Jupyter). Quita el "#" de la línea siguiente:
# %pip install -q imagehash pillow-heif scikit-image

In [ ]:
# ===== Importaciones =====
import os, io, json, random, copy, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix,
                             cohen_kappa_score)

# Soporte para fotos de iPhone (.heic), si la librería está instalada
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    print("Aviso: pillow-heif no está instalado; las fotos .heic no se podrán abrir.")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ===== Parámetros principales (todo lo "ajustable" vive aquí) =====
IMAGES_DIR = "Master_Vault"            # carpeta con las fotos originales
CSV_PATH   = "index_etiquetas.csv"     # planilla de etiquetas (se crea sola en el Paso 1)
OUT_DIR    = Path("eva_outputs")       # aquí se guardan modelos, splits y resultados
OUT_DIR.mkdir(exist_ok=True)

IMG_SIZE   = 320                       # tamaño de entrada de la red (px). 224 es el estándar; 320 conserva más detalle fino
CACHE_SIZE = int(IMG_SIZE * 1.15)      # tamaño al que se guardan las fotos en memoria (deja margen para recortes aleatorios)
SEED       = 42                        # semilla: hace que los resultados sean reproducibles
N_FOLDS    = 5                         # nº de repeticiones de la validación cruzada (también define el test: 1/5 = 20%)
BATCH_SIZE = 16
EPOCHS_HEAD     = 8                    # fase 1: entrenar solo la "cabeza" de la red
EPOCHS_FINETUNE = 12                   # fase 2: ajustar además los últimos bloques
LR_HEAD, LR_FINETUNE = 1e-3, 1e-4      # velocidad de aprendizaje de cada fase
PATIENCE   = 5                         # detener antes si no mejora en N épocas
HASH_DIST  = 6                         # sensibilidad para considerar dos fotos "casi duplicadas" (más bajo = más estricto)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de cómputo: {device}")

## Paso 1 · Crear la planilla (CSV) a partir de la carpeta de fotos — *lista todas las imágenes y deja columnas listas para etiquetar*

> **¿Qué hace?** Recorre la carpeta de fotos y arma (o actualiza) una planilla con una fila por foto. **Si ya existe la planilla, no borra lo que ya etiquetaste**: solo agrega las fotos nuevas. Puedes volver a ejecutar este bloque cada vez que lleguen fotos nuevas.
>
> **Columnas de la planilla:**
> - `filename`: nombre del archivo (con su extensión real).
> - `label`: **1** = hay helecho, **0** = no hay helecho, **vacío** = sin etiquetar / dudosa.
> - `quality`: `ok`, `borroso`, `sin_muestra`, `burbujas`, `otro`. (Vacío cuenta como `ok`.) Solo las fotos `ok` se usan para entrenar.
> - `group_id`: si sabes de qué usuaria viene la foto, anótalo anonimizado (`U001`, `U002`…). Sirve para que fotos de una misma persona no queden repartidas entre entrenamiento y test.
> - `source`: `whatsapp`, `instagram`, `otro` (si lo sabes).
> - `labeler`: quién etiquetó. `label_2`: etiqueta de una **segunda persona** (opcional, para medir acuerdo).
> - `notes`: observaciones libres.

In [ ]:
VALID_EXT = {".jpg", ".jpeg", ".png", ".webp", ".heic", ".heif", ".bmp", ".tif", ".tiff"}
COLUMNS = ["filename", "label", "quality", "group_id", "source", "labeler", "label_2", "notes"]

def load_square(path, size=CACHE_SIZE):
    """Abre una foto, corrige su rotación (EXIF), recorta el cuadrado central
    (el campo del microscopio suele estar centrado) y la reduce a size x size.
    Esta MISMA función se usa en entrenamiento y en inferencia, para que coincidan."""
    img = Image.open(path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    img = img.crop((left, top, left + s, top + s))
    return img.resize((size, size), Image.BICUBIC)

def construir_o_actualizar_csv():
    files = sorted(p.name for p in Path(IMAGES_DIR).iterdir() if p.suffix.lower() in VALID_EXT)
    if os.path.exists(CSV_PATH):
        d = pd.read_csv(CSV_PATH)
        for c in COLUMNS:
            if c not in d.columns:
                d[c] = np.nan
        nuevas = [f for f in files if f not in set(d.filename)]
        d = pd.concat([d, pd.DataFrame({"filename": nuevas})], ignore_index=True)
        faltantes = sorted(set(d.filename) - set(files))
        if faltantes:
            print(f"Aviso: {len(faltantes)} filas del CSV apuntan a archivos que ya no están en la carpeta.")
        print(f"CSV existente actualizado: {len(nuevas)} fotos nuevas agregadas.")
    else:
        d = pd.DataFrame({"filename": files})
        for c in COLUMNS[1:]:
            d[c] = np.nan
        print("CSV nuevo creado.")
    d[COLUMNS + [c for c in d.columns if c not in COLUMNS]].to_csv(CSV_PATH, index=False)
    return d

_d = construir_o_actualizar_csv()
print(f"Total de fotos en la planilla: {len(_d)} | ya etiquetadas: {_d.label.isin([0, 1]).sum()}")

## Paso 2 · Etiquetar las fotos (paso manual) — *aquí le enseñas al modelo qué es un helecho*

> **¿Qué hace?** Muestra "hojas de contacto" (varias fotos con su nombre) para que puedas etiquetar más rápido. **El etiquetado se hace editando el CSV** (en Excel, Google Sheets o similar) y guardándolo con el mismo nombre.
>
> **Recomendaciones (son lo que más pesa en la calidad del POC):**
> 1. **Duda = déjala vacía** (o márcala en `notes`). Es mejor excluir un caso ambiguo que forzar un 0/1 incierto.
> 2. Fotos inservibles (borrosas, sin muestra, con burbujas) → ponles el motivo en `quality`.
> 3. Etiqueta **sin mirar** quién la envió ni cuándo.
> 4. Pide a **una segunda persona** que etiquete ~100–200 fotos al azar en la columna `label_2`. Con eso se calcula el acuerdo (kappa) en el bloque siguiente.
>
> Usa `mostrar_hoja_de_contacto(inicio=0)` y avanza con `inicio=12`, `24`, … para ver las siguientes.

In [ ]:
def mostrar_hoja_de_contacto(inicio=0, n=12, columnas=4, solo_sin_etiqueta=True):
    """Muestra n fotos con su nombre de archivo para ayudar a etiquetar."""
    d = pd.read_csv(CSV_PATH)
    if solo_sin_etiqueta:
        d = d[~d.label.isin([0, 1])]
    d = d.iloc[inicio:inicio + n]
    if len(d) == 0:
        print("No hay más fotos para mostrar con ese filtro.")
        return
    filas = int(np.ceil(len(d) / columnas))
    fig, axes = plt.subplots(filas, columnas, figsize=(4 * columnas, 4 * filas))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, d.iterrows()):
        try:
            ax.imshow(load_square(Path(IMAGES_DIR) / r.filename, 300))
        except Exception as e:
            ax.text(0.5, 0.5, "no se pudo abrir", ha="center")
        ax.set_title(r.filename, fontsize=8)
    plt.tight_layout()
    plt.show()

mostrar_hoja_de_contacto(inicio=0)

### Paso 2b (opcional) · Acuerdo entre dos etiquetadores — *mide qué tan subjetiva es la tarea*

> **¿Qué hace?** Si hay una segunda persona que llenó `label_2`, calcula el **kappa de Cohen**: qué tanto coinciden ambas personas *descontando el azar*. Un kappa bajo indica que la tarea es muy subjetiva, y eso limita cuánto se le puede exigir al modelo. Es un número muy valioso para reportar.
>
> Guía: < 0.40 bajo · 0.40–0.60 moderado · 0.60–0.80 sustancial · > 0.80 casi perfecto.

In [ ]:
_d = pd.read_csv(CSV_PATH)
if "label_2" in _d.columns and _d.label_2.isin([0, 1]).any():
    ambos = _d[_d.label.isin([0, 1]) & _d.label_2.isin([0, 1])]
    if len(ambos) >= 20:
        kappa = cohen_kappa_score(ambos.label.astype(int), ambos.label_2.astype(int))
        acuerdo = (ambos.label == ambos.label_2).mean()
        print(f"Fotos etiquetadas por ambas personas: {len(ambos)}")
        print(f"Acuerdo simple: {acuerdo:.1%} | Kappa de Cohen: {kappa:.2f}")
        desacuerdos = ambos[ambos.label != ambos.label_2].filename.tolist()
        print(f"Fotos con desacuerdo ({len(desacuerdos)}), conviene revisarlas y definir una regla de desempate:")
        print(desacuerdos[:30])
    else:
        print(f"Aún hay pocas fotos con doble etiqueta ({len(ambos)}); se recomiendan al menos 100.")
else:
    print("Todavía no hay datos en 'label_2'. Este paso es opcional, pero muy recomendable.")

## Paso 3 · Cargar datos y revisarlos (EDA) — *mirar qué tenemos antes de entrenar*

> **¿Qué hace?**
> 1. Lee la planilla y se queda **solo con las fotos etiquetadas (0/1) y de calidad `ok`**.
> 2. Carga las imágenes a memoria (ya corregidas y recortadas).
> 3. Grafica la **distribución de clases** (¿cuántos helechos vs. no helechos?), el **tamaño original** de las fotos, y su **brillo** y **nitidez** por clase.
>
> **¿Por qué importa el brillo y la nitidez?** Si, por ejemplo, las fotos con helecho son sistemáticamente más brillantes o más nítidas que las otras, el modelo podría aprender *eso* en vez del patrón real. Si ves curvas muy distintas entre clases, anótalo como limitación.

In [ ]:
df_all = pd.read_csv(CSV_PATH)
df_all["quality"] = df_all["quality"].fillna("ok")

n_total = len(df_all)
n_sin_etiqueta = (~df_all.label.isin([0, 1])).sum()
n_no_ok = ((df_all.label.isin([0, 1])) & (df_all.quality != "ok")).sum()
df = df_all[df_all.label.isin([0, 1]) & (df_all.quality == "ok")].reset_index(drop=True)
df["label"] = df["label"].astype(int)
df["Clase"] = df.label.map({1: "Helecho (1)", 0: "No helecho (0)"})

print(f"Fotos en la planilla: {n_total}")
print(f"  - sin etiqueta / dudosas (excluidas): {n_sin_etiqueta}")
print(f"  - etiquetadas pero con calidad no 'ok' (excluidas): {n_no_ok}")
print(f"  - USADAS en este POC: {len(df)}")
print(df.Clase.value_counts().to_string())

if len(df) < 100:
    print("\n⚠️ Hay menos de 100 fotos utilizables: los resultados serán muy inestables.")
if df.label.value_counts().min() < 30:
    print("⚠️ La clase minoritaria tiene menos de 30 fotos: considera etiquetar más antes de interpretar métricas.")

In [ ]:
# ===== Cargar imágenes a memoria y calcular características simples =====
IMAGES = {}
tamanos = []
for fn in df.filename:
    p = Path(IMAGES_DIR) / fn
    with Image.open(p) as im:
        tamanos.append(im.size)
    IMAGES[fn] = load_square(p)

df["ancho"] = [t[0] for t in tamanos]
df["alto"] = [t[1] for t in tamanos]

def _nitidez(img):
    """Varianza del Laplaciano: más alto = más nítida."""
    g = np.asarray(img.convert("L"), dtype=np.float32)
    lap = -4 * g + np.roll(g, 1, 0) + np.roll(g, -1, 0) + np.roll(g, 1, 1) + np.roll(g, -1, 1)
    return float(lap.var())

df["brillo"] = [float(np.asarray(IMAGES[f].convert("L")).mean()) for f in df.filename]
df["nitidez"] = [_nitidez(IMAGES[f]) for f in df.filename]

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
sns.countplot(data=df, x="Clase", hue="Clase", palette="viridis", legend=False, ax=ax[0])
for c in ax[0].containers:
    ax[0].bar_label(c)
ax[0].set_title("Distribución de clases")
ax[1].scatter(df.ancho, df.alto, alpha=0.5)
ax[1].set_xlabel("Ancho original (px)"); ax[1].set_ylabel("Alto original (px)")
ax[1].set_title("Tamaño original de las fotos")
sns.histplot(data=df, x="brillo", hue="Clase", kde=True, ax=ax[2], palette="viridis")
ax[2].set_title("Brillo por clase")
sns.histplot(data=df, x="nitidez", hue="Clase", kde=True, ax=ax[3], palette="viridis", log_scale=True)
ax[3].set_title("Nitidez por clase (escala log)")
plt.tight_layout(); plt.show()

# Ejemplos de cada clase
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for fila, (lab, nombre) in enumerate([(1, "Helecho"), (0, "No helecho")]):
    muestra = df[df.label == lab].sample(min(6, (df.label == lab).sum()), random_state=SEED)
    for ax in axes[fila]:
        ax.axis("off")
    for ax, fn in zip(axes[fila], muestra.filename):
        ax.imshow(IMAGES[fn]); ax.set_title(nombre, fontsize=9)
plt.tight_layout(); plt.show()

### Paso 3b · Detectar fotos duplicadas o casi idénticas — *evitar que el modelo "copie" en vez de aprender*

> **¿Qué hace?** Calcula una "huella visual" (*hash perceptual*) de cada foto. Fotos que se parecen muchísimo (la misma imagen enviada por WhatsApp e Instagram, o varias tomas casi iguales de una misma gota) se agrupan en un mismo **grupo**. Si además llenaste `group_id` (usuaria), esas fotos también se unen.
>
> **¿Por qué?** En el Paso 4, **todos los miembros de un grupo irán juntos** al entrenamiento o al test, nunca separados. Así se evita la fuga de datos.
>
> **Qué revisar:** el tamaño del grupo más grande. Si un solo grupo concentra una parte enorme de las fotos, probablemente `HASH_DIST` es demasiado alto (agrupa fotos que solo se parecen por ser todas de saliva); bájalo y vuelve a ejecutar.

In [ ]:
import imagehash

hashes = [imagehash.phash(IMAGES[f]) for f in df.filename]
bits = np.array([h.hash.flatten() for h in hashes])
dist = (bits[:, None, :] != bits[None, :, :]).sum(axis=2)   # distancia entre huellas

def construir_grupos(df, dist, max_dist):
    """Une (union-find) fotos casi idénticas y fotos con el mismo group_id."""
    n = len(df)
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    ii, jj = np.where(np.triu(dist <= max_dist, k=1))
    for a, b in zip(ii, jj):
        union(a, b)
    if "group_id" in df.columns:
        for _, idx in df.groupby("group_id").indices.items():
            for k in idx[1:]:
                union(idx[0], k)
    return pd.factorize([find(i) for i in range(n)])[0]

df["group"] = construir_grupos(df, dist, HASH_DIST)
tam_grupos = df.group.value_counts()
print(f"Fotos: {len(df)} | grupos independientes: {df.group.nunique()}")
print(f"Grupos con más de 1 foto (posibles duplicados / misma usuaria): {(tam_grupos > 1).sum()}")
print(f"Tamaño del grupo más grande: {tam_grupos.max()} fotos ({tam_grupos.max() / len(df):.1%} del total)")
if tam_grupos.max() / len(df) > 0.10:
    print("⚠️ Un solo grupo concentra más del 10% de las fotos. Prueba bajando HASH_DIST.")

# Mostrar un ejemplo del grupo más grande (para verificar visualmente que son realmente duplicados)
g_top = tam_grupos.index[0]
if tam_grupos.iloc[0] > 1:
    ejemplo = df[df.group == g_top].filename.tolist()[:6]
    fig, axes = plt.subplots(1, len(ejemplo), figsize=(3 * len(ejemplo), 3))
    for ax, fn in zip(np.atleast_1d(axes), ejemplo):
        ax.imshow(IMAGES[fn]); ax.axis("off"); ax.set_title(fn, fontsize=7)
    plt.suptitle("Ejemplo del grupo más grande: ¿son realmente la misma muestra/foto?"); plt.show()

## Paso 4 · Separar los datos: entrenamiento vs. test congelado — *apartar un examen final que el modelo nunca verá*

> **¿Qué hace?**
> 1. Aparta ~20% de las fotos como **TEST CONGELADO**. No se usa para entrenar ni para tomar decisiones; solo se mira **una vez** en el Paso 7.
> 2. El 80% restante ("dev") se divide en **5 partes (folds)** para la validación cruzada del Paso 5.
> 3. Ambas divisiones respetan los **grupos** del paso anterior (sin fuga) y mantienen la proporción de helechos / no helechos (**estratificado**).
> 4. Guarda quién quedó en cada conjunto en `eva_outputs/splits.csv`, para poder reproducirlo.
>
> ⚠️ **Regla de oro:** no vuelvas a mirar el test para "ajustar" cosas. Si lo haces, deja de ser un test.

In [ ]:
idx_all = np.arange(len(df))

# --- 1) Test congelado: un fold de una división en N_FOLDS partes (= 20% si N_FOLDS=5)
sgkf_test = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds_test = list(sgkf_test.split(idx_all, df.label, groups=df.group))
test_idx = folds_test[0][1]
dev_idx = np.setdiff1d(idx_all, test_idx)

df["split"] = "dev"
df.loc[test_idx, "split"] = "test"
df["fold"] = -1

# --- 2) Validación cruzada dentro del conjunto dev
dev = df.loc[dev_idx]
sgkf_cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, va) in enumerate(sgkf_cv.split(dev.index, dev.label, groups=dev.group)):
    df.loc[dev.index[va], "fold"] = k

# --- Verificaciones de seguridad
g_test = set(df[df.split == "test"].group); g_dev = set(df[df.split == "dev"].group)
assert g_test.isdisjoint(g_dev), "¡Hay grupos repartidos entre dev y test!"
for k in range(N_FOLDS):
    g_va = set(df[df.fold == k].group); g_tr = set(df[(df.split == "dev") & (df.fold != k)].group)
    assert g_va.isdisjoint(g_tr), f"Fuga entre entrenamiento y validación en el fold {k}"
print("✔ Sin fuga de grupos entre entrenamiento, validación y test.\n")

resumen = df.groupby("split").label.agg(total="count", helechos="sum")
resumen["% helechos"] = (100 * resumen.helechos / resumen.total).round(1)
print(resumen)
print("\nHelechos por fold de validación (dentro de dev):")
print(df[df.split == "dev"].groupby("fold").label.agg(total="count", helechos="sum").T)

df[["filename", "label", "group", "split", "fold"]].to_csv(OUT_DIR / "splits.csv", index=False)
print(f"\nDivisión guardada en {OUT_DIR / 'splits.csv'}")

## Paso 5a · Preparar las imágenes para la red — *transformaciones y "aumento de datos"*

> **¿Qué hace?** Define cómo se le muestran las fotos a la red.
> - **Entrenamiento (con *data augmentation*):** cada vez que la red ve una foto, se la modifica un poco al azar (giro, recorte, rotación leve, desenfoque, compresión JPEG, brillo/contraste). Así "inventamos" variedad y simulamos lo que pasa en la vida real: fotos mal enfocadas o recomprimidas por WhatsApp/Instagram. Esto reduce el sobreajuste con pocos datos.
> - **Validación / test / uso real:** sin modificaciones aleatorias, solo redimensionar y normalizar con los valores de ImageNet (la red preentrenada los espera así).

In [ ]:
class RandomJPEG:
    """Simula la compresión de WhatsApp/Instagram re-guardando la imagen con calidad JPEG aleatoria."""
    def __init__(self, quality=(30, 90), p=0.5):
        self.quality, self.p = quality, p
    def __call__(self, img):
        if random.random() > self.p:
            return img
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(*self.quality))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),          # válido: en microscopía no hay "arriba" ni "abajo"
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
    RandomJPEG(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    NORM,
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORM,
])

class EVADataset(Dataset):
    """Entrega (imagen transformada, etiqueta) leyendo desde las imágenes ya cargadas en memoria."""
    def __init__(self, filenames, labels, images, transform):
        self.filenames, self.labels, self.images, self.transform = list(filenames), list(labels), images, transform
    def __len__(self):
        return len(self.filenames)
    def __getitem__(self, i):
        x = self.transform(self.images[self.filenames[i]])
        return x, torch.tensor(float(self.labels[i]), dtype=torch.float32)

def make_loader(sub, transform, shuffle):
    ds = EVADataset(sub.filename, sub.label, IMAGES, transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

# Vista previa: la misma foto con 6 versiones aumentadas distintas
fn0 = df.filename.iloc[0]
fig, axes = plt.subplots(1, 7, figsize=(20, 3))
axes[0].imshow(IMAGES[fn0]); axes[0].set_title("Original"); axes[0].axis("off")
inv = lambda t: (t * torch.tensor([0.229, 0.224, 0.225])[:, None, None] + torch.tensor([0.485, 0.456, 0.406])[:, None, None]).clamp(0, 1).permute(1, 2, 0).numpy()
for ax in axes[1:]:
    ax.imshow(inv(train_tf(IMAGES[fn0]))); ax.set_title("Aumentada"); ax.axis("off")
plt.show()

## Paso 5b · Definir el modelo y las funciones de entrenamiento — *MobileNetV3-Small preentrenado, en dos fases*

> **¿Qué hace?** Define (sin ejecutar todavía el entrenamiento) las piezas necesarias:
> - **`build_model`:** carga MobileNetV3-Small ya entrenada con ImageNet y cambia su última capa para que entregue **un solo número** (qué tan probable es "helecho").
> - **Entrenamiento en 2 fases:**
>   - *Fase 1 (cabeza):* se congela todo el "ojo" de la red y se entrena solo la parte final que decide. Es rápido y seguro.
>   - *Fase 2 (ajuste fino):* se liberan además los últimos 3 bloques, con velocidad de aprendizaje 10 veces menor, para que la red afine su visión a texturas de cristales.
> - **Corrección de BatchNorm:** las capas BatchNorm del modelo preentrenado se mantienen en modo "fijo", porque con pocas fotos recalcularlas empeora el modelo.
> - **`pos_weight`:** si hay menos helechos que no-helechos, se le da más peso a los errores en la clase minoritaria, para que no la ignore.
> - **Mejor época:** en cada fase se guarda el modelo de la época con mejor **AUC de validación** (no simplemente el último), y se detiene antes si deja de mejorar (*early stopping*).

In [ ]:
def build_model(pretrained=True):
    w = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.mobilenet_v3_small(weights=w)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 1)   # 1 salida: "puntaje de helecho"
    return m.to(device)

def set_trainable(model, phase):
    """phase='head': solo la cabeza. phase='finetune': cabeza + últimos 3 bloques del extractor."""
    for p in model.features.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    if phase == "finetune":
        for p in model.features[-3:].parameters():
            p.requires_grad = True

def set_train_mode(model):
    """Modo entrenamiento, pero con BatchNorm del extractor 'congelado' (usa estadísticas de ImageNet)."""
    model.train()
    for m in model.features.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

@torch.no_grad()
def predict_probs(model, loader):
    """Devuelve (probabilidades, etiquetas verdaderas) para todo un loader."""
    model.eval()
    P, Y = [], []
    for x, y in loader:
        P.append(torch.sigmoid(model(x.to(device))).cpu().numpy().ravel())
        Y.append(y.numpy().ravel())
    return np.concatenate(P), np.concatenate(Y)

def run_phase(model, train_loader, val_loader, criterion, epochs, lr, phase, state, fold):
    set_trainable(model, phase)
    opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    sin_mejora = 0
    for ep in range(epochs):
        set_train_mode(model)
        tot, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device).unsqueeze(1)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
            tot += loss.item() * len(x); n += len(x)
        sched.step()
        vp, vy = predict_probs(model, val_loader)
        vauc = roc_auc_score(vy, vp) if len(np.unique(vy)) > 1 else float("nan")
        state["hist"].append(dict(fold=fold, phase=phase, epoch=len(state["hist"]) + 1,
                                  train_loss=tot / n, val_auc=vauc))
        print(f"  [{phase:8s}] época {ep+1:2d}/{epochs} | pérdida entren.: {tot/n:.4f} | AUC validación: {vauc:.3f}")
        if np.nan_to_num(vauc, nan=-1) > state["best_auc"]:
            state["best_auc"], state["best_state"], sin_mejora = vauc, copy.deepcopy(model.state_dict()), 0
        else:
            sin_mejora += 1
            if sin_mejora >= PATIENCE:
                print("  (detenido antes: sin mejora)")
                break

def train_one_fold(k, tr, va):
    set_seed(SEED + k)
    train_loader, val_loader = make_loader(tr, train_tf, True), make_loader(va, val_tf, False)
    n_pos, n_neg = int(tr.label.sum()), int((tr.label == 0).sum())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / max(n_pos, 1)], device=device))
    model = build_model()
    state = dict(best_auc=-1.0, best_state=None, hist=[])
    run_phase(model, train_loader, val_loader, criterion, EPOCHS_HEAD, LR_HEAD, "head", state, k)
    model.load_state_dict(state["best_state"])                      # partir de lo mejor de la fase 1
    run_phase(model, train_loader, val_loader, criterion, EPOCHS_FINETUNE, LR_FINETUNE, "finetune", state, k)
    model.load_state_dict(state["best_state"])
    torch.save(state["best_state"], OUT_DIR / f"model_fold{k}.pt")
    vp, vy = predict_probs(model, val_loader)
    return model, pd.DataFrame(state["hist"]), vp

## Paso 5c · Entrenar con validación cruzada — *entrenar 5 modelos, cada uno "examinado" con un grupo distinto*

> **¿Qué hace?** Entrena 5 veces el modelo (una por fold). En cada vuelta, 4 partes del conjunto dev sirven para entrenar y la quinta para validar. Al final, **cada foto de dev tiene una predicción hecha por un modelo que nunca la vio** (*predicciones out-of-fold*), lo que da una estimación más confiable que un único split.
>
> **¿Cuánto demora?** Con GPU, pocos minutos; con CPU puede tardar bastante más (conviene bajar `IMG_SIZE` a 224 para una primera prueba).
>
> **Cómo leer lo que imprime:** busca que el *AUC de validación* suba por encima de 0.5 (azar). Si se queda cerca de 0.5, el modelo no está encontrando señal.
>
> *Nota técnica:* como se elige la mejor época usando el mismo fold de validación, el AUC de CV es levemente optimista. La cifra realmente imparcial es la del test congelado (Paso 7).

In [ ]:
dev = df[df.split == "dev"].copy()
oof_parts, hist_parts, fold_aucs = [], [], []

t0 = time.time()
for k in range(N_FOLDS):
    print(f"\n===== Fold {k+1}/{N_FOLDS} =====")
    tr, va = dev[dev.fold != k], dev[dev.fold == k]
    model_k, hist_k, vp = train_one_fold(k, tr, va)
    hist_parts.append(hist_k)
    oof_parts.append(pd.DataFrame({"filename": va.filename.values, "label": va.label.values,
                                   "group": va.group.values, "fold": k, "prob": vp}))
    fold_aucs.append(roc_auc_score(va.label, vp))
    print(f"  → AUC del fold {k}: {fold_aucs[-1]:.3f}")
print(f"\nTiempo total: {(time.time() - t0)/60:.1f} min")

oof = pd.concat(oof_parts, ignore_index=True)
history = pd.concat(hist_parts, ignore_index=True)
oof.to_csv(OUT_DIR / "oof_predictions.csv", index=False)

# Curvas de aprendizaje por fold
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, h in history.groupby("fold"):
    ax[0].plot(h.epoch, h.train_loss, label=f"fold {k}")
    ax[1].plot(h.epoch, h.val_auc, label=f"fold {k}")
ax[0].set_title("Pérdida de entrenamiento (debe bajar)"); ax[0].set_xlabel("Época acumulada")
ax[1].set_title("AUC en validación (debe subir)"); ax[1].set_xlabel("Época acumulada")
ax[1].axhline(0.5, color="gray", ls="--"); ax[1].legend()
plt.tight_layout(); plt.show()

### Paso 5d · Resultados de la validación cruzada — *cómo le fue al modelo "examinado" en datos que no vio*

> **¿Qué hace?** Resume el desempeño con las predicciones out-of-fold: AUC global y por fold (promedio ± desviación), y calcula el **umbral de decisión** óptimo.
>
> **¿Qué es el umbral?** El modelo entrega un puntaje entre 0 y 1; hay que decidir desde qué valor se declara "helecho". Como entrenamos con `pos_weight`, el 0.5 ya no es necesariamente lo mejor, así que se elige el umbral con el criterio de **Youden** (maximiza sensibilidad + especificidad), **usando solo datos de dev**, nunca el test.

In [ ]:
def metricas_binarias(y, p, thr):
    pred = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    div = lambda a, b: a / b if b else float("nan")
    return dict(umbral=thr, sensibilidad=div(tp, tp + fn), especificidad=div(tn, tn + fp),
                VPP=div(tp, tp + fp), VPN=div(tn, tn + fn), exactitud=div(tp + tn, len(y)))

auc_oof = roc_auc_score(oof.label, oof.prob)
fpr, tpr, ths = roc_curve(oof.label, oof.prob)
THR = float(ths[np.argmax(tpr - fpr)])

print(f"AUC global (out-of-fold): {auc_oof:.3f}")
print(f"AUC por fold: {np.round(fold_aucs, 3)}  →  {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")
print(f"\nUmbral elegido (Youden, solo con datos dev): {THR:.3f}")
print(pd.DataFrame([metricas_binarias(oof.label, oof.prob, 0.5),
                    metricas_binarias(oof.label, oof.prob, THR)],
                   index=["umbral 0.5", "umbral Youden"]).round(3).to_string())

## Paso 6 · Línea base clásica (LBP + regresión logística) — *¿una técnica simple lograría lo mismo?*

> **¿Qué hace?** Entrena un método "de la vieja escuela" que no usa redes neuronales: describe la **textura** de cada foto con histogramas **LBP** (*Local Binary Patterns*) y usa una **regresión logística** para clasificar. Se evalúa con **exactamente los mismos folds**.
>
> **¿Para qué sirve?** Da contexto al resultado de la CNN. Si la CNN supera claramente a este método simple, justifica su complejidad; si rinden parecido, es información importante para el diseño del estudio real (¡y un método así sería mucho más barato de correr!).

In [ ]:
try:
    from skimage.feature import local_binary_pattern
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    HAS_LBP = True
except ImportError:
    HAS_LBP = False
    print("scikit-image no está instalado; se omite la línea base (pip install scikit-image).")

def lbp_features(img):
    g = np.asarray(img.convert("L").resize((256, 256)))
    feats = []
    for P, R in [(8, 1), (16, 2), (24, 3)]:
        lbp = local_binary_pattern(g, P, R, "uniform")
        h, _ = np.histogram(lbp, bins=P + 2, range=(0, P + 2), density=True)
        feats.append(h)
    return np.concatenate(feats)

def nuevo_clasificador_lbp():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))

if HAS_LBP:
    X_lbp = np.array([lbp_features(IMAGES[f]) for f in df.filename])
    oof_lbp = np.zeros(len(df))
    for k in range(N_FOLDS):
        tr_i = df.index[(df.split == "dev") & (df.fold != k)]
        va_i = df.index[(df.split == "dev") & (df.fold == k)]
        clf = nuevo_clasificador_lbp().fit(X_lbp[tr_i], df.label[tr_i])
        oof_lbp[va_i] = clf.predict_proba(X_lbp[va_i])[:, 1]
    mask_dev = (df.split == "dev").values
    auc_lbp = roc_auc_score(df.label[mask_dev], oof_lbp[mask_dev])
    print(f"AUC out-of-fold  |  LBP + regresión logística: {auc_lbp:.3f}   vs.   MobileNet: {auc_oof:.3f}")

## Paso 7 · Evaluación final sobre el TEST CONGELADO (¡una sola vez!) — *el examen final, con márgenes de error*

> **¿Qué hace?**
> 1. Junta los 5 modelos entrenados (**ensamble**: el puntaje final es el promedio de los 5) y los prueba sobre las fotos del test, que ninguno vio.
> 2. Muestra **matriz de confusión** (aciertos y tipos de error), **curva ROC** y **curva Precisión-Sensibilidad**, comparando con la línea base.
> 3. Calcula **intervalos de confianza al 95%** por *bootstrap por grupos*: se repite el cálculo 1000 veces re-muestreando grupos de fotos, para ver cuánto podrían variar las métricas por pura suerte de la muestra.
> 4. Guarda la predicción de cada foto de test en `eva_outputs/test_predictions.csv` y la configuración final.
>
> **Cómo leer los intervalos:** con ~100 fotos de test los intervalos serán **anchos**, y eso es normal y honesto. Reporta siempre el rango, no solo el número central.
>
> ⚠️ Ejecuta este bloque **una vez**, cuando ya hayas terminado de ajustar todo lo demás.

In [ ]:
test = df[df.split == "test"].copy()
test_loader = make_loader(test, val_tf, False)

ENSEMBLE, probs_modelos = [], []
for k in range(N_FOLDS):
    m = build_model(pretrained=False)
    m.load_state_dict(torch.load(OUT_DIR / f"model_fold{k}.pt", map_location=device))
    m.eval()
    ENSEMBLE.append(m)
    p, y_test = predict_probs(m, test_loader)
    probs_modelos.append(p)

p_test = np.mean(probs_modelos, axis=0)
y_test = y_test.astype(int)
g_test_arr = test.group.values
print(f"Fotos en el test: {len(test)} ({y_test.sum()} helechos, {(y_test == 0).sum()} no helechos)\n")

# ----- Métricas puntuales
auc_test = roc_auc_score(y_test, p_test)
ap_test = average_precision_score(y_test, p_test)
tabla = pd.DataFrame([metricas_binarias(y_test, p_test, 0.5), metricas_binarias(y_test, p_test, THR)],
                     index=["umbral 0.5", f"umbral Youden (de dev)"]).round(3)
print(f"AUC en test: {auc_test:.3f} | PR-AUC: {ap_test:.3f}")
print(tabla.to_string())

# ----- Bootstrap por grupos (IC 95%)
def bootstrap_ci(y, p, groups, thr, n_boot=1000):
    rng = np.random.default_rng(SEED)
    ug = np.unique(groups)
    idx_por_g = {g: np.where(groups == g)[0] for g in ug}
    res = {"AUC": [], "sensibilidad": [], "especificidad": []}
    for _ in range(n_boot):
        sel = rng.choice(ug, len(ug), replace=True)
        idx = np.concatenate([idx_por_g[g] for g in sel])
        yy, pp = y[idx], p[idx]
        if len(np.unique(yy)) < 2:
            continue
        res["AUC"].append(roc_auc_score(yy, pp))
        pred = pp >= thr
        res["sensibilidad"].append((pred & (yy == 1)).sum() / (yy == 1).sum())
        res["especificidad"].append((~pred & (yy == 0)).sum() / (yy == 0).sum())
    base = metricas_binarias(y, p, thr)
    base["AUC"] = roc_auc_score(y, p)
    return pd.DataFrame({k: [base[k], *np.percentile(v, [2.5, 97.5])] for k, v in res.items()},
                        index=["estimación", "IC95% inf", "IC95% sup"]).T.round(3)

print(f"\nIntervalos de confianza al 95% (bootstrap por grupos, umbral = {THR:.3f}):")
ic_tabla = bootstrap_ci(y_test, p_test, g_test_arr, THR)
print(ic_tabla.to_string())

# ----- Gráficos
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
cm = confusion_matrix(y_test, (p_test >= THR).astype(int), labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=["No helecho", "Helecho"], yticklabels=["No helecho", "Helecho"])
ax[0].set_xlabel("Predicho por el modelo"); ax[0].set_ylabel("Etiqueta real"); ax[0].set_title("Matriz de confusión (test)")

fpr_t, tpr_t, _ = roc_curve(y_test, p_test)
ax[1].plot(fpr_t, tpr_t, lw=2, label=f"MobileNet (AUC={auc_test:.2f})")
if HAS_LBP:
    clf_full = nuevo_clasificador_lbp().fit(X_lbp[df.split == "dev"], df.label[df.split == "dev"])
    p_lbp_test = clf_full.predict_proba(X_lbp[test.index])[:, 1]
    f2, t2, _ = roc_curve(y_test, p_lbp_test)
    ax[1].plot(f2, t2, lw=2, label=f"LBP+LogReg (AUC={roc_auc_score(y_test, p_lbp_test):.2f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=1)
ax[1].set_xlabel("1 - Especificidad (falsos positivos)"); ax[1].set_ylabel("Sensibilidad")
ax[1].set_title("Curva ROC (test)"); ax[1].legend(loc="lower right")

pr, rc, _ = precision_recall_curve(y_test, p_test)
ax[2].plot(rc, pr, lw=2); ax[2].axhline(y_test.mean(), color="gray", ls="--", label="azar (prevalencia)")
ax[2].set_xlabel("Sensibilidad"); ax[2].set_ylabel("Precisión (VPP)"); ax[2].set_title("Precisión-Sensibilidad (test)")
ax[2].legend(); plt.tight_layout(); plt.show()

# ----- Guardar resultados
test_pred = pd.DataFrame({"filename": test.filename.values, "label": y_test, "prob": p_test,
                          "pred": (p_test >= THR).astype(int), "group": g_test_arr})
test_pred["correcto"] = test_pred.label == test_pred.pred
test_pred.to_csv(OUT_DIR / "test_predictions.csv", index=False)
ic_tabla.to_csv(OUT_DIR / "test_metrics_ci.csv")
with open(OUT_DIR / "config.json", "w") as f:
    json.dump({"img_size": IMG_SIZE, "cache_size": CACHE_SIZE, "umbral": THR, "n_modelos": N_FOLDS,
               "auc_cv_oof": auc_oof, "auc_test": auc_test}, f, indent=2)
print(f"\nResultados guardados en {OUT_DIR}/")

## Paso 8 · ¿Qué está mirando el modelo? (Grad-CAM) — *comprobar que mira cristales y no "trampas"*

> **¿Qué hace?** Para ejemplos del test (aciertos y errores) dibuja un **mapa de calor** sobre la foto: las zonas **rojas/amarillas** son las que más influyeron en la decisión del modelo.
>
> **¿Por qué es importante?** Con fotos espontáneas, el modelo podría "hacer trampa": fijarse en bordes del campo, burbujas, reflejos o suciedad en vez de los cristales. Si los mapas apuntan a las estructuras de helecho, es una buena señal. Si apuntan a los bordes o al fondo, hay que desconfiar de las métricas.
>
> **Filas:** verdaderos positivos (acertó helecho) · verdaderos negativos (acertó no-helecho) · **falsos positivos** (dijo helecho y no lo era) · **falsos negativos** (se le escapó un helecho). Los errores son lo más informativo: ¿tienen algo en común (borrosas, burbujas, helecho parcial)?

In [ ]:
def gradcam(model, x):
    """Devuelve (mapa de calor 0-1 de tamaño IMG_SIZE, puntaje del modelo) para una imagen ya transformada."""
    model.eval()
    for p in model.parameters():
        p.requires_grad_(True)
    x = x.to(device)
    with torch.enable_grad():
        feats = model.features(x)
        feats.retain_grad()
        out = model.classifier(torch.flatten(model.avgpool(feats), 1))
        model.zero_grad()
        out.sum().backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * feats).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, torch.sigmoid(out).item()

modelo_cam = ENSEMBLE[int(np.argmax(fold_aucs))]     # usamos el modelo del mejor fold
tp = test_pred[(test_pred.label == 1) & (test_pred.pred == 1)].sort_values("prob", ascending=False)
tn = test_pred[(test_pred.label == 0) & (test_pred.pred == 0)].sort_values("prob")
fp = test_pred[(test_pred.label == 0) & (test_pred.pred == 1)].sort_values("prob", ascending=False)
fn = test_pred[(test_pred.label == 1) & (test_pred.pred == 0)].sort_values("prob")
categorias = [("Acierto: helecho", tp), ("Acierto: no helecho", tn),
              ("ERROR: falso positivo", fp), ("ERROR: falso negativo", fn)]

N_COL = 4
fig, axes = plt.subplots(len(categorias), N_COL, figsize=(3.6 * N_COL, 3.6 * len(categorias)))
for ax in axes.ravel():
    ax.axis("off")
for fila, (nombre, sub) in enumerate(categorias):
    for col, (_, r) in enumerate(sub.head(N_COL).iterrows()):
        img = IMAGES[r.filename]
        cam, score = gradcam(modelo_cam, val_tf(img).unsqueeze(0))
        axes[fila, col].imshow(img.resize((IMG_SIZE, IMG_SIZE)))
        axes[fila, col].imshow(cam, cmap="jet", alpha=0.40)
        axes[fila, col].set_title(f"{nombre}\n{r.filename[:22]} | puntaje {score:.2f}", fontsize=8)
    if len(sub) == 0:
        axes[fila, 0].text(0.0, 0.5, f"{nombre}: sin casos", transform=axes[fila, 0].transAxes)
plt.tight_layout(); plt.show()

## Paso 9 · Probar con una foto nueva — *simula el uso real: una usuaria envía una foto*

> **¿Qué hace?** Define `predecir_nueva_foto(ruta)`, que procesa la foto **exactamente igual** que en el entrenamiento, la pasa por los 5 modelos y devuelve un resultado.
>
> **Detalles pensados para evitar malas interpretaciones:**
> - El resultado dice **"patrón compatible con cristalización"**, no "fértil".
> - Se muestra un **puntaje del modelo**, *no* un porcentaje de confianza: estos puntajes no están calibrados, así que 0.80 **no** significa "80% de probabilidad".
> - Si el puntaje cae cerca del umbral, el resultado es **"Indeterminado"** y se sugiere repetir la foto, en vez de forzar una respuesta.
> - Se informa si los 5 modelos discrepan entre sí (señal de foto atípica).
>
> **Limitación importante:** el modelo no sabe reconocer fotos inservibles (sin muestra, muy borrosas). Siempre dará una respuesta, así que en una versión real habría que añadir un filtro de calidad.

In [ ]:
def cargar_ensemble():
    """Recarga los modelos y el umbral desde disco (útil para usar el modelo sin reentrenar)."""
    cfg = json.load(open(OUT_DIR / "config.json"))
    modelos = []
    for k in range(cfg["n_modelos"]):
        m = build_model(pretrained=False)
        m.load_state_dict(torch.load(OUT_DIR / f"model_fold{k}.pt", map_location=device))
        m.eval(); modelos.append(m)
    return modelos, cfg["umbral"]

def predecir_nueva_foto(ruta, ensemble=None, umbral=None, banda_incierta=0.15, mostrar=True):
    ensemble = ensemble if ensemble is not None else ENSEMBLE
    umbral = umbral if umbral is not None else THR
    try:
        img = load_square(ruta)
    except FileNotFoundError:
        print(f"No se encontró la imagen: {ruta}")
        return None
    x = val_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        puntajes = [torch.sigmoid(m(x)).item() for m in ensemble]
    p, desacuerdo = float(np.mean(puntajes)), float(np.std(puntajes))

    if p >= umbral + banda_incierta:
        veredicto = "Patrón compatible con cristalización (helecho)"
    elif p <= umbral - banda_incierta:
        veredicto = "No se observa patrón de helecho"
    else:
        veredicto = "Indeterminado: se recomienda repetir la foto"

    if mostrar:
        plt.figure(figsize=(5, 5.4))
        plt.imshow(img); plt.axis("off")
        plt.title(f"{veredicto}\nPuntaje del modelo: {p:.2f} (umbral {umbral:.2f})\n"
                  f"Discrepancia entre modelos: {desacuerdo:.2f}", fontsize=9)
        plt.show()
    return {"veredicto": veredicto, "puntaje": p, "umbral": umbral, "discrepancia_modelos": desacuerdo}

# Ejemplo de uso (reemplaza por la ruta de una foto nueva):
# resultado = predecir_nueva_foto("/ruta/a/foto_nueva_whatsapp.jpg")
# print(resultado)

## Cierre · Cómo presentar estos resultados — *plantilla para el informe o la postulación*

**Qué reportar (siempre con intervalos de confianza):**
- Nº de fotos recibidas, nº utilizadas, distribución de clases, nº de grupos independientes.
- Kappa entre etiquetadores (si se hizo el Paso 2b).
- AUC, sensibilidad y especificidad **en el test congelado**, con IC95% (tabla del Paso 7).
- Comparación con la línea base clásica (Paso 6) y un par de ejemplos de Grad-CAM (Paso 8), incluyendo errores.

**Cómo interpretar según lo que salga:**
- *AUC ≈ 0.5:* no se encontró señal; revisar calidad de etiquetas y cantidad de datos antes de concluir nada.
- *AUC bueno en CV pero claramente peor en test:* sospechar sobreajuste, fuga residual de datos o test muy pequeño (mirar el ancho del IC).
- *Grad-CAM apuntando a bordes/fondo:* el modelo probablemente usa atajos; las métricas no son confiables.
- *LBP rinde parecido a la CNN:* la señal es simple de capturar; útil para diseñar el estudio real.

**Limitaciones a declarar explícitamente:**
1. Fotos espontáneas, sin protocolo de captura (tiempo de secado, iluminación, enfoque, grosor de la gota).
2. Sin metadatos de dispositivo, y usuaria/canal solo parcialmente conocidos.
3. Posible sesgo de autoselección (quien comparte es quien vio algo llamativo).
4. Etiquetado por pocas personas; la clasificación de ferning es subjetiva.
5. **Sin ground truth clínico:** el modelo detecta *patrones visuales*, no ovulación ni fertilidad.
6. Muestra pequeña: los intervalos de confianza son amplios.

**Plan v2 (lo que este POC enseña que hay que hacer):**
- Protocolo de captura estandarizado + formulario con metadatos (dispositivo, momento del ciclo, hora, ayuno, etc.).
- Consentimiento informado explícito y anonimización desde el origen; conservar archivos originales sin recomprimir.
- Doble etiquetado independiente, escala ordinal (0 / parcial / completo) y clase "no evaluable".
- Test **externo** (usuarias y dispositivos nunca vistos).
- Estudio prospectivo con una referencia objetiva (p. ej., tests de LH, temperatura basal) para pasar de "detecta helechos" a "sirve para algo".